# Next Experiment: Multi-Window CT and Segmentation-Guided Learning

The previous experiments showed a clear limitation: **sensitivity remains relatively low**, with the latest test results reaching **61.29% sensitivity, 47.20% F1, and 86.62% specificity**. This made me question whether the model is missing hemorrhages because it lacks complementary information or because it is not learning enough about **where the hemorrhage is**.

The next question is:

> **Can additional CT information and explicit localization supervision help the model detect the cases it currently misses?**

I will therefore explore:

* **Multiple CT windows**, including brain and bone.
* **Hemorrhage segmentation masks as training supervision**.
* A **multi-task model** that learns both ICH classification and localization.

Rather than assuming these additions will improve performance, I want to see **what actually changes in the model's behavior**, particularly its sensitivity and spatial focus.


# Data

Each slice will use:
- **Brain window**
- **Bone window**
- **ICH diagnosis** from `hemorrhage_diagnosis.csv`
- **Hemorrhage segmentation mask** as the localization target

The CT windows will be the model inputs, while the diagnosis and mask will be used as training targets.


In [63]:
from pathlib import Path

import pandas as pd
from PIL import Image

DATA_DIR = Path("data")


diagnosis_path = DATA_DIR / "hemorrhage_diagnosis.csv"

if not diagnosis_path.exists():
    raise FileNotFoundError(f"Diagnosis file not found: {diagnosis_path}")

diagnosis_df = pd.read_csv(diagnosis_path)

diagnosis_df.head()

,PatientNumber,SliceNumber,Intraventricular,Intraparenchymal,Subarachnoid,Epidural,Subdural,No_Hemorrhage,Fracture_Yes_No
0,49,1,0,0,0,0,0,1,0
1,49,2,0,0,0,0,0,1,0
2,49,3,0,0,0,0,0,1,0
3,49,4,0,0,0,0,0,1,0
4,49,5,0,0,0,0,0,1,0


In [64]:
def load_ct_windows(
    patient_id: int,
    slice_number: int,
) -> tuple[Image.Image, Image.Image]:
    """Load brain- and bone-window CT images for one slice."""

    patient_dir = DATA_DIR / "Patients_CT" / f"{patient_id:03d}"

    brain_path = patient_dir / "brain" / f"{slice_number}.jpg"
    bone_path = patient_dir / "bone" / f"{slice_number}.jpg"

    if not brain_path.exists():
        raise FileNotFoundError(f"Brain-window image not found: {brain_path}")

    if not bone_path.exists():
        raise FileNotFoundError(f"Bone-window image not found: {bone_path}")

    brain_image = Image.open(brain_path).convert("L")
    bone_image = Image.open(bone_path).convert("L")

    return brain_image, bone_image

## Loading Check

The brain and bone windows are both available and have the same spatial resolution.

Example: Patient 49, Slice 1 → **650 × 650** for both windows.

In [65]:
sample = diagnosis_df.iloc[0]

patient_id = 49
slice_number = 17

brain_image, bone_image = load_ct_windows(
    patient_id=patient_id,
    slice_number=slice_number,
)

print(f"Patient: {patient_id}")
print(f"Slice: {slice_number}")
print(f"Brain size: {brain_image.size}")
print(f"Bone size: {bone_image.size}")

Patient: 49
Slice: 17
Brain size: (650, 650)
Bone size: (650, 650)


## Segmentation Masks

Hemorrhage segmentation masks are available only for selected slices containing annotated hemorrhage.

For a slice such as `17.jpg`, the corresponding mask is `17_HGE_Seg.jpg` when an annotation is available.

Slices without a mask will be treated as having **no segmentation annotation**, rather than as missing data.

In [66]:
def load_segmentation_mask(
    patient_id: int,
    slice_number: int,
) -> Image.Image | None:
    """Load the hemorrhage segmentation mask if available."""

    mask_path = (
        DATA_DIR
        / "Patients_CT"
        / f"{patient_id:03d}"
        / "brain"
        / f"{slice_number}_HGE_Seg.jpg"
    )

    if not mask_path.exists():
        return None

    return Image.open(mask_path).convert("L")

In [67]:
mask_image = load_segmentation_mask(
    patient_id=patient_id,
    slice_number=slice_number,
)

if mask_image is not None:
    print(f"Mask size: {mask_image.size}")
else:
    print("No segmentation mask available for this slice.")

Mask size: (650, 650)


## Diagnosis and Mask Consistency

The diagnosis dataframe provides the **ICH label for each slice**.

The presence of a segmentation file indicates that a localization annotation is available. These two sources will be checked together to distinguish:
- ICH slices with segmentation annotations
- ICH slices without segmentation annotations
- Non-ICH slices without segmentation annotations

Any unexpected combination will be flagged for inspection.

## Dataset Construction

Each sample will combine the **brain-window image**, **bone-window image**, and **ICH diagnosis** for a given slice.

When available, the corresponding **segmentation mask** will also be associated with the sample.

Segmentation annotations will remain optional so that slices without masks can still be used for classification.

In [68]:
diagnosis_df["ICH"] = 1 - diagnosis_df["No_Hemorrhage"]

dataset_records = []

for _, row in diagnosis_df.iterrows():
    patient_id = int(row["PatientNumber"])
    slice_number = int(row["SliceNumber"])

    brain_path = (
        DATA_DIR / "Patients_CT" / f"{patient_id:03d}" / "brain" / f"{slice_number}.jpg"
    )

    bone_path = (
        DATA_DIR / "Patients_CT" / f"{patient_id:03d}" / "bone" / f"{slice_number}.jpg"
    )

    mask_path = (
        DATA_DIR
        / "Patients_CT"
        / f"{patient_id:03d}"
        / "brain"
        / f"{slice_number}_HGE_Seg.jpg"
    )

    dataset_records.append(
        {
            "patient_id": patient_id,
            "slice_number": slice_number,
            "brain_path": brain_path,
            "bone_path": bone_path,
            "mask_path": mask_path if mask_path.exists() else None,
            "label": int(row["ICH"]),
        }
    )

dataset_df = pd.DataFrame(dataset_records)

## Train/Test Split

The same **patient-level split** used in the previous experiments will be retained.

Patients are divided into:
- **Development set** for training and validation
- **Test set** for final evaluation

All slices from a patient remain in the same split to prevent patient-level data leakage and keep the results comparable with the previous model.

In [69]:
import json

from src.preprocessing import compute_class_weights, create_patient_labels

# Patient-level split
patient_labels = create_patient_labels(diagnosis_df)

split_path = Path("data/static_patient_split.json")

with open(split_path, "r") as f:
    split = json.load(f)

development_patients = patient_labels.loc[
    patient_labels["PatientNumber"].isin(split["development"])
]

test_patients = patient_labels.loc[patient_labels["PatientNumber"].isin(split["test"])]

In [70]:
development_patient_ids = set(development_patients["PatientNumber"])
test_patient_ids = set(test_patients["PatientNumber"])

development_df = dataset_df[
    dataset_df["patient_id"].isin(development_patient_ids)
].reset_index(drop=True)

test_df = dataset_df[dataset_df["patient_id"].isin(test_patient_ids)].reset_index(
    drop=True
)

In [71]:
print(f"Development: {len(development_df)} slices")
print(f"Test: {len(test_df)} slices")

print(f"Development patients: {development_df['patient_id'].nunique()}")
print(f"Test patients: {test_df['patient_id'].nunique()}")

Development: 1983 slices
Test: 518 slices
Development patients: 65
Test patients: 17


## Data Consistency

In [72]:
missing_brain = []
missing_bone = []

for _, row in diagnosis_df.iterrows():
    patient_id = int(row["PatientNumber"])
    slice_number = int(row["SliceNumber"])

    patient_dir = DATA_DIR / "Patients_CT" / f"{patient_id:03d}"

    brain_path = patient_dir / "brain" / f"{slice_number}.jpg"
    bone_path = patient_dir / "bone" / f"{slice_number}.jpg"

    if not brain_path.exists():
        missing_brain.append((patient_id, slice_number))

    if not bone_path.exists():
        missing_bone.append((patient_id, slice_number))

print(f"Missing brain images: {len(missing_brain)}")
print(f"Missing bone images: {len(missing_bone)}")

if missing_bone:
    print("Missing bone images:", missing_bone)

Missing brain images: 0
Missing bone images: 1
Missing bone images: [(84, 36)]


In [73]:
valid_window_pairs = development_df["brain_path"].apply(
    lambda p: p.exists()
) & development_df["bone_path"].apply(lambda p: p.exists())

print(f"Development slices before filtering: {len(development_df)}")
print(f"Incomplete pairs: {(~valid_window_pairs).sum()}")

development_df = development_df[valid_window_pairs].reset_index(drop=True)

print(f"Development slices after filtering: {len(development_df)}")

valid_test_pairs = test_df["brain_path"].apply(lambda p: p.exists()) & test_df[
    "bone_path"
].apply(lambda p: p.exists())

print(f"Test slices before filtering: {len(test_df)}")
print(f"Incomplete pairs: {(~valid_test_pairs).sum()}")

test_df = test_df[valid_test_pairs].reset_index(drop=True)

print(f"Test slices after filtering: {len(test_df)}")

Development slices before filtering: 1983
Incomplete pairs: 1
Development slices after filtering: 1982
Test slices before filtering: 518
Incomplete pairs: 0
Test slices after filtering: 518


# Preprocessing

The brain and bone windows will be used as separate input channels. The segmentation mask corresponds to the **hemorrhage region in the brain-window CT** and will be used as the localization target.

Spatial transformations will be applied jointly to both CT windows and the segmentation mask, using **bilinear interpolation for CT images** and **nearest-neighbor interpolation for the mask**. This preserves the spatial correspondence between the two CT windows and the hemorrhage localization target while avoiding artificial values in the mask.

The two CT channels will be **normalized separately** because brain and bone windows have different intensity distributions. Statistics will be computed from the development set only to avoid using information from the test set.

Segmentation masks will be converted to binary tensors. Slices without hemorrhage will use an all-zero mask, while slices without an available segmentation annotation will be handled separately during training.

## Window Statistics

Brain and bone windows have different intensity distributions, so their normalization statistics will be computed separately.

Only the **development set** will be used to compute these statistics, keeping the test set completely unseen during preprocessing.

In [74]:
import numpy as np


def compute_image_stats(image_paths):
    """Compute mean and standard deviation across grayscale images."""

    pixel_sum = 0.0
    pixel_sum_sq = 0.0
    pixel_count = 0

    for path in image_paths:
        image = (
            np.asarray(
                Image.open(path).convert("L"),
                dtype=np.float32,
            )
            / 255.0
        )

        pixel_sum += image.sum()
        pixel_sum_sq += (image**2).sum()
        pixel_count += image.size

    mean = pixel_sum / pixel_count
    variance = (pixel_sum_sq / pixel_count) - (mean**2)
    std = np.sqrt(variance)

    return mean, std

In [75]:
brain_mean, brain_std = compute_image_stats(development_df["brain_path"])

bone_mean, bone_std = compute_image_stats(development_df["bone_path"])

print(f"Brain: mean={brain_mean:.4f}, std={brain_std:.4f}")
print(f"Bone:  mean={bone_mean:.4f}, std={bone_std:.4f}")

Brain: mean=0.2098, std=0.3261
Bone:  mean=0.1347, std=0.1720


In [76]:
import albumentations as A
import cv2
import torch
from albumentations.pytorch import ToTensorV2


def get_multiwindow_train_transforms(
    brain_mean: float,
    brain_std: float,
    bone_mean: float,
    bone_std: float,
):
    """Preprocessing and synchronized augmentation for brain + bone CT."""

    return A.Compose(
        [
            A.Resize(
                height=224,
                width=224,
                interpolation=cv2.INTER_LINEAR,
            ),
            A.HorizontalFlip(p=0.5),
            A.Rotate(
                limit=8,
                interpolation=cv2.INTER_LINEAR,
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                fill_mask=0,
                p=0.5,
            ),
            A.Affine(
                translate_percent=(0.05, 0.05),
                interpolation=cv2.INTER_LINEAR,
                mask_interpolation=cv2.INTER_NEAREST,
                border_mode=cv2.BORDER_CONSTANT,
                fill=0,
                fill_mask=0,
                p=0.5,
            ),
            A.RandomBrightnessContrast(
                brightness_limit=0.1,
                contrast_limit=0.1,
                p=0.5,
            ),
            A.Normalize(
                mean=[brain_mean, bone_mean],
                std=[brain_std, bone_std],
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )


def get_multiwindow_eval_transforms(
    brain_mean: float,
    brain_std: float,
    bone_mean: float,
    bone_std: float,
):
    """Deterministic preprocessing for validation and testing."""

    return A.Compose(
        [
            A.Resize(
                height=224,
                width=224,
                interpolation=cv2.INTER_LINEAR,
            ),
            A.Normalize(
                mean=[brain_mean, bone_mean],
                std=[brain_std, bone_std],
                max_pixel_value=255.0,
            ),
            ToTensorV2(),
        ],
    )

## Dataset Construction

Each sample combines the **brain and bone CT windows** with its ICH diagnosis and, when available, a hemorrhage segmentation mask.

The dataset returns the two CT windows as a **2-channel input**, together with the classification label and segmentation target. For slices without a segmentation annotation, the mask is kept as a placeholder and an availability flag is returned so that segmentation supervision can be ignored during training.


In [77]:
from torch.utils.data import Dataset


class MultiWindowCTDataset(Dataset):
    """Dataset for brain + bone CT with optional ICH segmentation targets."""

    def __init__(
        self,
        dataframe: pd.DataFrame,
        transform=None,
    ):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        brain = np.array(
            Image.open(row["brain_path"]).convert("L"),
            dtype=np.uint8,
        )

        bone = np.array(
            Image.open(row["bone_path"]).convert("L"),
            dtype=np.uint8,
        )

        label = int(row["label"])

        mask_available = pd.notna(row["mask_path"])

        if mask_available:
            mask = np.array(
                Image.open(row["mask_path"]).convert("L"),
                dtype=np.uint8,
            )
            mask = (mask > 0).astype(np.uint8)
        else:
            mask = np.zeros_like(brain, dtype=np.uint8)

        if self.transform is not None:
            transformed = self.transform(
                image=np.stack([brain, bone], axis=-1),
                mask=mask,
            )

            image = transformed["image"]
            mask = transformed["mask"]
        else:
            image = torch.from_numpy(np.stack([brain, bone], axis=0)).float() / 255.0

            mask = torch.from_numpy(mask).float()

        if mask.ndim == 2:
            mask = mask.unsqueeze(0)

        mask = (mask > 0.5).float()

        return {
            "image": image,
            "label": torch.tensor(label, dtype=torch.long),
            "mask": mask,
            "mask_available": torch.tensor(
                mask_available,
                dtype=torch.bool,
            ),
            "patient_id": int(row["patient_id"]),
            "slice_number": int(row["slice_number"]),
        }

In [78]:
train_transforms = get_multiwindow_train_transforms(
    brain_mean=brain_mean,
    brain_std=brain_std,
    bone_mean=bone_mean,
    bone_std=bone_std,
)

eval_transforms = get_multiwindow_eval_transforms(
    brain_mean=brain_mean,
    brain_std=brain_std,
    bone_mean=bone_mean,
    bone_std=bone_std,
)

train_dataset = MultiWindowCTDataset(
    development_df,
    transform=train_transforms,
)

In [79]:
sample = train_dataset[0]

for key, value in sample.items():
    if isinstance(value, torch.Tensor):
        print(f"{key}: shape={tuple(value.shape)}, " f"dtype={value.dtype}")
    else:
        print(f"{key}: {value}")

print("\nImage range:", sample["image"].min().item(), sample["image"].max().item())
print("Label:", sample["label"].item())
print("Mask values:", torch.unique(sample["mask"]))
print("Mask available:", sample["mask_available"].item())

image: shape=(2, 224, 224), dtype=torch.float32
label: shape=(), dtype=torch.int64
mask: shape=(1, 224, 224), dtype=torch.float32
mask_available: shape=(), dtype=torch.bool
patient_id: 49
slice_number: 1

Image range: -0.7830227017402649 5.031078815460205
Label: 0
Mask values: tensor([0.])
Mask available: False


# Model

The model will use a shared **ResNet-18 encoder** to process the brain and bone windows jointly.

Two prediction heads will then be used:

* a **classification head** for ICH detection
* a **segmentation head** for hemorrhage localization

The classification and segmentation tasks will share the learned representation, allowing the localization task to provide additional spatial supervision during training.

The model will therefore learn:

**Brain + Bone → Shared Features → ICH Classification + Hemorrhage Segmentation**


In [80]:
from torch import nn
from torchvision.models import ResNet18_Weights, resnet18


class MultiWindowMultiTaskModel(nn.Module):
    """
    ResNet-18 based multi-task model for brain + bone CT.

    Outputs:
        classification_logits: [B, 2]
        segmentation_logits: [B, 1, H, W]
    """

    def __init__(self, pretrained=True):
        super().__init__()

        weights = ResNet18_Weights.DEFAULT if pretrained else None
        backbone = resnet18(weights=weights)

        # Adapt the first convolution from 3 input channels to 2.
        old_conv = backbone.conv1

        new_conv = nn.Conv2d(
            in_channels=2,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False,
        )

        with torch.no_grad():
            # Average the pretrained RGB filters.
            pretrained_weights = old_conv.weight
            mean_weights = pretrained_weights.mean(dim=1, keepdim=True)

            new_conv.weight.copy_(mean_weights.repeat(1, 2, 1, 1))

        backbone.conv1 = new_conv

        # Shared ResNet feature extractor.
        self.stem = nn.Sequential(
            backbone.conv1,
            backbone.bn1,
            backbone.relu,
            backbone.maxpool,
        )

        self.layer1 = backbone.layer1
        self.layer2 = backbone.layer2
        self.layer3 = backbone.layer3
        self.layer4 = backbone.layer4

        # Classification head.
        self.classifier_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.classifier = nn.Linear(
            512,
            2,
        )

        # Segmentation decoder.
        self.segmentation_decoder = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, kernel_size=1),
        )

    def forward(self, x):
        # Shared encoder.
        x = self.stem(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        features = self.layer4(x)

        # Classification branch.
        pooled = self.classifier_pool(features)
        pooled = torch.flatten(pooled, 1)

        classification_logits = self.classifier(pooled)

        # Segmentation branch.
        segmentation_logits = self.segmentation_decoder(features)

        segmentation_logits = nn.functional.interpolate(
            segmentation_logits,
            size=(224, 224),
            mode="bilinear",
            align_corners=False,
        )

        return classification_logits, segmentation_logits


model = MultiWindowMultiTaskModel(pretrained=True)

In [81]:
sample_input = sample["image"].unsqueeze(0)

with torch.no_grad():
    classification_logits, segmentation_logits = model(sample_input)

print("Input:", sample_input.shape)
print("Classification:", classification_logits.shape)
print("Segmentation:", segmentation_logits.shape)

Input: torch.Size([1, 2, 224, 224])
Classification: torch.Size([1, 2])
Segmentation: torch.Size([1, 1, 224, 224])


# Training

The model will be trained using **4-fold patient-level cross-validation** on the development set.

The training setup is kept consistent with the previous experiment where possible:
- batch size: 32
- learning rate: $10^{-4}$
- AdamW optimizer
- weight decay: $10^{-4}$, $10^{-3}$, and $10^{-2}$
- ReduceLROnPlateau scheduler
- weighted classification loss
- weighted random sampling

The main difference is that the model now optimizes both **ICH classification** and **hemorrhage segmentation**.

The total training loss is:

$$
L = L_{classification} + \lambda L_{segmentation}
$$

where $\lambda = 1.0$ for this experiment.

Validation **F1 score** is used for model checkpointing, since the main objective remains ICH detection rather than segmentation performance.

## 4-Fold Cross-Validation

Patient-level folds are used to ensure that slices from the same patient never appear in both training and validation sets.

For each fold, the model is trained on three patient groups and validated on the remaining group. The process is repeated for each weight decay value.

The best epoch is selected using validation F1, with early stopping after 5 consecutive epochs without improvement.

In [ ]:
from sklearn.model_selection import StratifiedKFold
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader, WeightedRandomSampler

BATCH_SIZE = 32
NUM_EPOCHS = 20
EARLY_STOPPING_PATIENCE = 5

LEARNING_RATE = 1e-4

SCHEDULER_FACTOR = 0.5
SCHEDULER_PATIENCE = 2

NUM_WORKERS = 4

WEIGHT_DECAY_VALUES = [
    1e-4,
    1e-3,
    1e-2,
]

SEGMENTATION_LOSS_WEIGHT = 1.0


def create_multiwindow_patient_folds(
    patient_labels: pd.DataFrame,
    n_splits: int = 4,
    random_state: int = 42,
):
    """Create stratified patient-level folds."""
    splitter = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=random_state,
    )

    folds = []

    patient_ids = patient_labels["PatientNumber"].to_numpy()
    labels = patient_labels["ICH"].to_numpy()

    for fold_number, (train_idx, validation_idx) in enumerate(
        splitter.split(patient_ids, labels),
        start=1,
    ):
        train_patients = patient_labels.iloc[train_idx].reset_index(drop=True)
        validation_patients = patient_labels.iloc[
            validation_idx
        ].reset_index(drop=True)

        folds.append({
            "fold": fold_number,
            "train": train_patients,
            "validation": validation_patients,
        })

    return folds

In [83]:
multiwindow_patient_labels = development_patients[["PatientNumber", "ICH"]].copy()

multiwindow_folds = create_multiwindow_patient_folds(
    multiwindow_patient_labels,
    n_splits=4,
    random_state=42,
)

for fold_info in multiwindow_folds:
    print(
        f"Fold {fold_info['fold']}: "
        f"{len(fold_info['train'])} train patients, "
        f"{len(fold_info['validation'])} validation patients"
    )

Fold 1: 48 train patients, 17 validation patients
Fold 2: 49 train patients, 16 validation patients
Fold 3: 49 train patients, 16 validation patients
Fold 4: 49 train patients, 16 validation patients


## Multi-Task Loss

The model has two objectives: classify each slice as ICH or Non-ICH and localize the hemorrhage when a segmentation annotation is available.

The classification loss uses the same class weighting strategy as the previous experiment.

For segmentation, BCE and Dice losses are combined to account for the sparse hemorrhage regions.

Slices without an available segmentation annotation are excluded from the segmentation loss rather than being treated as negative examples.

In [ ]:
def compute_segmentation_loss(
    segmentation_logits: torch.Tensor,
    masks: torch.Tensor,
    mask_available: torch.Tensor,
):
    """Compute BCE + Dice loss for slices with available masks."""
    if not mask_available.any():
        return segmentation_logits.sum() * 0.0

    logits = segmentation_logits[mask_available]
    targets = masks[mask_available]

    bce_loss = nn.functional.binary_cross_entropy_with_logits(
        logits,
        targets,
    )

    probabilities = torch.sigmoid(logits)

    intersection = (probabilities * targets).sum(dim=(1, 2, 3))
    denominator = probabilities.sum(dim=(1, 2, 3)) + targets.sum(dim=(1, 2, 3))

    dice_loss = 1.0 - ((2.0 * intersection + 1e-6) / (denominator + 1e-6)).mean()

    return bce_loss + dice_loss


def compute_multitask_loss(
    classification_logits,
    segmentation_logits,
    labels,
    masks,
    mask_available,
    class_weights,
    segmentation_weight=1.0,
):
    """Compute weighted classification + segmentation loss."""
    classification_loss = nn.functional.cross_entropy(
        classification_logits,
        labels,
        weight=class_weights,
    )

    segmentation_loss = compute_segmentation_loss(
        segmentation_logits,
        masks,
        mask_available,
    )

    total_loss = classification_loss + segmentation_weight * segmentation_loss

    return total_loss, classification_loss, segmentation_loss

## Training Loop

The training loop optimizes the combined classification and segmentation loss at each step.

Validation is performed after each epoch. The best checkpoint is selected using validation F1, while early stopping limits unnecessary training when the validation score stops improving.

In [ ]:
from copy import deepcopy

from sklearn.metrics import f1_score


def train_multiwindow_model(
    model,
    train_loader,
    validation_loader,
    class_weights,
    segmentation_weight,
    optimizer,
    scheduler,
    device,
    num_epochs,
    patience,
    checkpoint_metric="f1",
):
    """Train the multi-task model with classification-based checkpointing."""

    best_metric = -float("inf")
    best_model_state = None
    best_validation_predictions = None
    best_epoch = 0
    epochs_without_improvement = 0

    history = []
    epoch_states = []

    for epoch in range(num_epochs):
        model.train()

        train_total_loss = 0.0
        train_classification_loss = 0.0
        train_segmentation_loss = 0.0

        for batch in train_loader:
            images = batch["image"].to(device)
            labels = batch["label"].to(device)
            masks = batch["mask"].to(device)
            mask_available = batch["mask_available"].to(device)

            optimizer.zero_grad()

            classification_logits, segmentation_logits = model(images)

            total_loss, classification_loss, segmentation_loss = compute_multitask_loss(
                classification_logits=classification_logits,
                segmentation_logits=segmentation_logits,
                labels=labels,
                masks=masks,
                mask_available=mask_available,
                class_weights=class_weights,
                segmentation_weight=segmentation_weight,
            )

            total_loss.backward()
            optimizer.step()

            train_total_loss += total_loss.item()
            train_classification_loss += classification_loss.item()
            train_segmentation_loss += segmentation_loss.item()

        train_total_loss /= len(train_loader)
        train_classification_loss /= len(train_loader)
        train_segmentation_loss /= len(train_loader)

        model.eval()

        validation_loss = 0.0
        validation_predictions = []
        validation_labels = []

        with torch.no_grad():
            for batch in validation_loader:
                images = batch["image"].to(device)
                labels = batch["label"].to(device)
                masks = batch["mask"].to(device)
                mask_available = batch["mask_available"].to(device)

                classification_logits, segmentation_logits = model(images)

                total_loss, _, _ = compute_multitask_loss(
                    classification_logits=classification_logits,
                    segmentation_logits=segmentation_logits,
                    labels=labels,
                    masks=masks,
                    mask_available=mask_available,
                    class_weights=class_weights,
                    segmentation_weight=segmentation_weight,
                )

                validation_loss += total_loss.item()

                predictions = torch.argmax(
                    classification_logits,
                    dim=1,
                )

                validation_predictions.extend(predictions.cpu().numpy())
                validation_labels.extend(labels.cpu().numpy())

        validation_loss /= len(validation_loader)

        validation_f1 = f1_score(
            validation_labels,
            validation_predictions,
            zero_division=0,
        )

        scheduler.step(validation_loss)

        epoch_states.append(
            {
                "epoch": epoch + 1,
                "train_total_loss": train_total_loss,
                "train_classification_loss": train_classification_loss,
                "train_segmentation_loss": train_segmentation_loss,
                "validation_loss": validation_loss,
                "validation_f1": validation_f1,
                "learning_rate": optimizer.param_groups[0]["lr"],
            }
        )

        history.append(
            {
                "epoch": epoch + 1,
                "train_total_loss": train_total_loss,
                "train_classification_loss": train_classification_loss,
                "train_segmentation_loss": train_segmentation_loss,
                "validation_loss": validation_loss,
                "validation_f1": validation_f1,
            }
        )

        print(
            f"Epoch {epoch + 1:02d}/{num_epochs} | "
            f"Train Loss: {train_total_loss:.4f} | "
            f"Val Loss: {validation_loss:.4f} | "
            f"Val F1: {validation_f1:.4f}"
        )

        if checkpoint_metric == "f1":
            current_metric = validation_f1
        else:
            raise ValueError(f"Unsupported checkpoint metric: {checkpoint_metric}")

        if current_metric > best_metric:
            best_metric = current_metric
            best_model_state = deepcopy(model.state_dict())
            best_validation_predictions = {
                "labels": np.asarray(validation_labels),
                "predictions": np.asarray(validation_predictions),
            }
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch + 1}. " f"Best epoch: {best_epoch}")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return (
        model,
        history,
        best_validation_predictions,
        epoch_states,
        best_epoch,
    )



## Cross-Validation Training

Each fold uses the same patient-level split for all weight decay values.

Training samples use the synchronized multi-window augmentation, while validation samples use deterministic preprocessing.

A weighted random sampler is used to address the class imbalance in the classification task. The model checkpoint is selected using validation F1.

In [ ]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = device.type == "cuda"

fold_results = {weight_decay: {} for weight_decay in WEIGHT_DECAY_VALUES}

for weight_decay in WEIGHT_DECAY_VALUES:
    print(f"\n{'#' * 60}")
    print(f"Weight decay: {weight_decay}")
    print(f"{'#' * 60}")

    for fold in multiwindow_folds:
        fold_number = fold["fold"]

        print(f"\n{'=' * 20} Fold {fold_number} {'=' * 20}")

        train_patient_ids = fold["train"]["PatientNumber"].tolist()
        validation_patient_ids = fold["validation"]["PatientNumber"].tolist()

        train_dataset = MultiWindowCTDataset(
            dataframe=development_df[
                development_df["patient_id"].isin(train_patient_ids)
            ].reset_index(drop=True),
            transform=train_transforms,
        )

        validation_dataset = MultiWindowCTDataset(
            dataframe=development_df[
                development_df["patient_id"].isin(validation_patient_ids)
            ].reset_index(drop=True),
            transform=eval_transforms,
        )

        train_labels = train_dataset.dataframe["label"].to_numpy()

        class_sample_counts = np.bincount(train_labels)

        sample_weights = np.array(
            [1.0 / class_sample_counts[label] for label in train_labels]
        )

        sampler = WeightedRandomSampler(
            weights=torch.as_tensor(
                sample_weights,
                dtype=torch.double,
            ),
            num_samples=len(train_dataset),
            replacement=True,
        )

        train_loader = DataLoader(
            train_dataset,
            batch_size=BATCH_SIZE,
            sampler=sampler,
            num_workers=NUM_WORKERS,
            pin_memory=pin_memory,
        )

        validation_loader = DataLoader(
            validation_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=pin_memory,
        )

        model = MultiWindowMultiTaskModel(
            pretrained=True,
        ).to(device)

        class_weights = compute_class_weights(train_labels).to(device)

        optimizer = AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=weight_decay,
        )

        scheduler = ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=SCHEDULER_FACTOR,
            patience=SCHEDULER_PATIENCE,
        )

        (
            model,
            history,
            best_validation_predictions,
            epoch_states,
            best_epoch,
        ) = train_multiwindow_model(
            model=model,
            train_loader=train_loader,
            validation_loader=validation_loader,
            class_weights=class_weights,
            segmentation_weight=SEGMENTATION_LOSS_WEIGHT,
            optimizer=optimizer,
            scheduler=scheduler,
            device=device,
            num_epochs=NUM_EPOCHS,
            patience=EARLY_STOPPING_PATIENCE,
            checkpoint_metric="f1",
        )

        fold_results[weight_decay][fold_number] = {
            "model": model,
            "history": history,
            "best_validation_predictions": best_validation_predictions,
            "epoch_states": epoch_states,
            "best_epoch": best_epoch,
            "train_patient_ids": train_patient_ids,
            "validation_patient_ids": validation_patient_ids,
        }